## Autograd

In [50]:
import math

In [51]:
class Tensor:
    def __init__(self, data, label='', _children=()):
        self.data = data
        self.grad = 0.0
        self.label = label
        self._prev = set(_children)
        self._backward = lambda: None

    def __repr__(self):
        return (
            f"Tensor(data={self.data}, "
            f"grad={self.grad}, "
            f"label='{self.label}')"
        )

    def __add__(self, other):
        if not isinstance(other, Tensor):
            other = Tensor(other)

        out = Tensor(
            self.data + other.data,
            _children=(self, other),
            label='+'
        )

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward

        return out

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        out = Tensor(
            -self.data,
            _children=(self,),
            label='neg'
        )

        def _backward():
            self.grad += -out.grad

        out._backward = _backward

        return out

    def __sub__(self, other):
        if not isinstance(other, Tensor):
            other = Tensor(other)

        return self + (-other)

    def __rsub__(self, other):
        if not isinstance(other, Tensor):
            other = Tensor(other)

        return other - self

    def __mul__(self, other):
        if not isinstance(other, Tensor):
            other = Tensor(other)

        out = Tensor(
            self.data * other.data,
            _children=(self, other),
            label='*'
        )

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward

        return out

    def __rmul__(self, other):
        return self * other

    def __pow__(self, power):
        out = Tensor(
            self.data ** power,
            _children=(self,),
            label=f'**{power}'
        )

        def _backward():
            self.grad += (
                power
                * self.data ** (power - 1)
                * out.grad
            )

        out._backward = _backward

        return out

    def relu(self):
        out = Tensor(
            max(0, self.data),
            _children=(self,),
            label='relu'
        )

        def _backward():
            self.grad += (out.data > 0) * out.grad

        out._backward = _backward

        return out

    def tanh(self):
        t = math.tanh(self.data)

        out = Tensor(
            t,
            _children=(self,),
            label='tanh'
        )

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward

        return out

    def sigmoid(self):
        x = self.data

        if x >= 0:
            s = 1 / (1 + math.exp(-x))
        else:
            exp_x = math.exp(x)
            s = exp_x / (1 + exp_x)

        out = Tensor(
            s,
            _children=(self,),
            label='sigmoid'
        )

        def _backward():
            self.grad += s * (1 - s) * out.grad

        out._backward = _backward

        return out

    def log(self):
        out = Tensor(
            math.log(self.data),
            _children=(self,),
            label='log'
        )

        def _backward():
            self.grad += (1 / self.data) * out.grad

        out._backward = _backward

        return out

    def backward(self):
        self.grad = 1.0

        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)

                for child in v._prev:
                    build_topo(child)

                topo.append(v)

        build_topo(self)

        for v in reversed(topo):
            v._backward()


### Neural network

In [52]:
import random

In [53]:
class Neuron:
    def __init__(self, nin, nonlin=True, activation=None):
        self.w = [Tensor(random.uniform(-1, 1), label=f'w{i}') for i in range(nin)]
        self.b = Tensor(0)
        self.nonlin = nonlin
        self.activation = activation

    def __call__(self, x):
        n = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        if self.activation == "sigmoid":
            return n.sigmoid()

        if self.nonlin:
            return n.relu()
        return n
    
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, n_inputs, n_neurons, nonlin=True, activation=None):
        self.neurons = [Neuron(n_inputs, nonlin, activation) for _ in range(n_neurons)]

    def __call__(self, x):
        out = [neuron(x) for neuron in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = []

        for i in range(len(nouts)):
            if i < len(nouts) - 1:
                layer = Layer(sz[i], sz[i + 1], nonlin=True)
            else:
                layer = Layer(sz[i], sz[i + 1], nonlin=False, activation="sigmoid")
            self.layers.append(layer)

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
